這邊改用深度學習模型: FFN

訓練優化：

使用AdamW優化器
	實現了早停機制(early stopping)
	批次處理(batch processing)提高效率
	自動GPU支持

特徵處理：
	保留了原有的文本和實體嵌入特徵
	添加了特徵標準化和正則化

評估指標：
	仍然使用AUC作為主要評估指標
	增加了訓練過程中的損失監控
	
硬體規格: 記憶體32gb，GPU rtx4060 16G 

嵌入模型: 'BAAI/bge-base-en-v1.5' 

num_workers=0

加入 Optuna 進行優化

# 載入套件 & 定義參數與儲存資料路徑

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import os
import pickle
from tqdm.notebook import tqdm
import logging
from datetime import datetime
import sys
import optuna # 匯入 Optuna
import shutil # 匯入 shutil 用於檔案操作

# --- 配置路徑 ---
DATA_DIR = '../data/' # 主資料目錄
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR = os.path.join(DATA_DIR, 'test')

TRAIN_BEHAVIORS_PATH = os.path.join(TRAIN_DIR, 'train_behaviors.tsv')
TEST_BEHAVIORS_PATH = os.path.join(TEST_DIR, 'test_behaviors.tsv')

TRAIN_NEWS_PATH = os.path.join(TRAIN_DIR, 'train_news.tsv')
TEST_NEWS_PATH = os.path.join(TEST_DIR, 'test_news.tsv')

RESULT_DIR = '../result'

# 更新嵌入模型名稱和快取路徑以反映新模型
# BAAI/bge-small-en-v1.5 ; NEWS_EMBEDDING_DIM: 384
# BAAI/bge-base-en-v1.5  ; NEWS_EMBEDDING_DIM: 768
NEWS_EMBEDDING_MODEL_NAME = 'BAAI/bge-base-en-v1.5' # 使用較小的模型
NEWS_EMBEDDINGS_CACHE_PATH = os.path.join(RESULT_DIR, f'news_embeddings_{NEWS_EMBEDDING_MODEL_NAME.replace("/", "_")}.pkl') # 動態生成快取檔案名
MODEL_SAVE_PATH = os.path.join(RESULT_DIR, 'best_model_auc_deep_learning.pth') # 模型儲存路徑可以更具體
LOG_FILE_PATH = os.path.join(RESULT_DIR, 'log_deep_learning_optuna.txt') # 礬土中文註解：更新日誌檔名以反映 Optuna 的使用
SUBMISSION_PATH = os.path.join(RESULT_DIR, 'submission_deep_learning_optuna.csv') # 礬土中文註解：更新提交檔名

# 模型與訓練參數
NEWS_EMBEDDING_DIM = 768  # 384: bge-small-en-v1.5 的嵌入維度 ; 768: BAAI/bge-base-en-v1.5
USER_HISTORY_EMBEDDING_DIM = NEWS_EMBEDDING_DIM
MODEL_INPUT_DIM = USER_HISTORY_EMBEDDING_DIM + NEWS_EMBEDDING_DIM

EPOCHS_PER_TRIAL = 30 # 設定每個 Optuna trial 的 epoch 數量 (原為 EPOCHS)
N_OPTUNA_TRIALS = 20 #  設定 Optuna 的試驗次數
EARLY_STOPPING_PATIENCE = 5
VALIDATION_SPLIT_RATIO = 0.2
RANDOM_SEED = 42


os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

# --- 設定日誌 ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE_PATH, mode='a', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


# --- 設定隨機種子以確保可重複性 ---
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    logger.info(f"PyTorch CUDA version: {torch.version.cuda}")
    logger.info(f"CUDA device count: {torch.cuda.device_count()}")
    current_device_id = torch.cuda.current_device()
    logger.info(f"Current CUDA device: {torch.cuda.get_device_name(current_device_id)}")

    # 啟用 cuDNN benchmark 模式，可能加速訓練
    torch.backends.cudnn.enabled = True
    torch.backends.cudnn.benchmark = True
    logger.info("cuDNN benchmark is enabled.")
else:
    logger.info("CUDA is not available. Using CPU.")


# 設定硬體設備 CPU 或 GPU 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {DEVICE}") # 使用 logger 保持一致性


2025-06-06 18:25:02,679 - INFO - PyTorch CUDA version: 11.8
2025-06-06 18:25:02,680 - INFO - CUDA device count: 1
2025-06-06 18:25:02,683 - INFO - Current CUDA device: NVIDIA GeForce RTX 4060 Ti
2025-06-06 18:25:02,683 - INFO - cuDNN benchmark is enabled.
2025-06-06 18:25:02,684 - INFO - Using device: cuda


# 資料載入與初步檢視

In [2]:
# --- 載入new資料集 ---
def load_news_data(file_path):
    """載入新聞資料並進行初步清理"""
    logger.info(f"開始載入新聞資料: {file_path}")
    try:
        df = pd.read_csv(file_path, sep='\t', header=None,
                         names=['news_id', 'category', 'subcategory', 'title', 'abstract', 'url',
                                'title_entities', 'abstract_entities'], skiprows=1)
        for col in ['title', 'abstract', 'category', 'subcategory', 'title_entities', 'abstract_entities']:
            df[col] = df[col].fillna('')
        logger.info(f"新聞資料 {os.path.basename(file_path)} 載入完成，共 {len(df)} 筆記錄。")
        if len(df) < 50000: # 只對較小的資料印出詳細資訊，避免日誌過長
            logger.debug(f"新聞資料前5筆:\n{df.head()}")
            logger.debug(f"新聞資料缺失值情況:\n{df.isnull().sum()}")
        return df
    except FileNotFoundError:
        logger.error(f"錯誤: 檔案 {file_path} 未找到。請檢查路徑。")
        return None
    except Exception as e:
        logger.error(f"載入新聞資料 {file_path} 時發生錯誤: {e}")
        return None

# --- 載入behaviors資料集 ---
def load_behaviors_data(file_path, is_train=True): 
    """載入行為資料，使用指定欄位名稱"""
    logger.info(f"開始載入行為資料: {file_path}")
    try:
        # if is_train:
        col_names = ['id', 'user_id', 'time', 'clicked_news_history', 'impressions']
        # else: 
        #     # 測試集的 behaviors.tsv 通常沒有 clicked_news_history，或者說不是我們關心的
        #     # 但題目給的 test_behaviors.tsv 其實是 *有* clicked_news_history 欄位的，只是我們要預測的是 impressions 的點擊
        #     # 根據原始碼，test_behaviors.tsv 確實讀取了 clicked_news_history
        #     col_names = ['id', 'user_id', 'time', 'clicked_news_history', 'impressions']

        df = pd.read_csv(file_path, sep='\t', header=None, names=col_names, skiprows=1)
        logger.info(f"行為資料 {os.path.basename(file_path)} 載入完成，共 {len(df)} 筆記錄。")
        if len(df) < 300000: # 只對較小的資料印出詳細資訊，避免日誌過長
            logger.debug(f"行為資料前5筆:\n{df.head()}")
            logger.debug(f"行為資料缺失值情況:\n{df.isnull().sum()}")

        # clicked_news_history 在 train 時重要，test 時也可能用到 (如果用戶歷史用於推斷)
        df['clicked_news_history'] = df['clicked_news_history'].fillna('')
        df['impressions'] = df['impressions'].fillna('')
        return df
    except FileNotFoundError:
        logger.error(f"錯誤: 檔案 {file_path} 未找到。請檢查路徑。")
        return None
    except Exception as e:
        logger.error(f"載入行為資料 {file_path} 時發生錯誤: {e}")
        return None


# 創建news特徵工程
使用SentenceTransformer轉換成embedding vector 

In [ ]:
# --- 取得new資料集的embeddings vector ---
# 如果cache存在   - 直接讀取，並檢查維度是否正確
# 如果cache不存在 - generate_fresh_embeddings() 使用SentenceTransformer 生成 embeddings vector
def generate_news_embeddings_with_bge(news_df, model_name=NEWS_EMBEDDING_MODEL_NAME, cache_path=NEWS_EMBEDDINGS_CACHE_PATH):
    if os.path.exists(cache_path):
        logger.info(f"從快取檔案載入新聞嵌入: {cache_path}")
        try:
            with open(cache_path, 'rb') as f:
                news_embeddings_dict = pickle.load(f)
            
            # 驗證快取維度是否正確
            if news_embeddings_dict:
                sample_news_id = next(iter(news_embeddings_dict))
                if news_embeddings_dict[sample_news_id].shape[0] == NEWS_EMBEDDING_DIM:
                    logger.info(f"快取的新聞嵌入維度 ({news_embeddings_dict[sample_news_id].shape[0]}) 正確。")
                    return news_embeddings_dict
                else:
                    logger.warning(f"快取的新聞嵌入維度 ({news_embeddings_dict[sample_news_id].shape[0]}) 與預期 ({NEWS_EMBEDDING_DIM}) 不符。將重新生成。")
            else:
                logger.warning(f"快取檔案 {cache_path} 為空。將重新生成。")
        except Exception as e:
            logger.error(f"載入快取檔案 {cache_path} 失敗: {e}. 將重新生成。")
            
    return generate_fresh_embeddings(news_df, model_name, cache_path)

# --- 生成new資料集的embeddings vector ---
def generate_fresh_embeddings(news_df, model_name, cache_path):
    global DEVICE # 允許在 CPU 失敗時修改 DEVICE
    logger.info(f"開始使用 {model_name} 生成新聞嵌入。這可能需要一些時間...")
    try:
        # 載入 SentenceTransformer model
        model = SentenceTransformer(model_name, device=DEVICE)
        logger.info(f"SentenceTransformer 模型 {model_name} 已成功載入到 {DEVICE}。")
    except Exception as e:
        logger.error(f"在 {DEVICE} 上載入 SentenceTransformer 模型 {model_name} 失敗: {e}.")
        if DEVICE != torch.device('cpu'): # 嘗試在 CPU 上載入
            logger.warning("嘗試在 CPU 上載入模型...")
            try:
                DEVICE = torch.device('cpu') # 修改全域 DEVICE
                model = SentenceTransformer(model_name, device=DEVICE)
                logger.info(f"模型已在 CPU 上成功載入。後續 PyTorch 裝置也將使用 CPU。")
            except Exception as e_cpu:
                logger.error(f"在 CPU 上載入 SentenceTransformer 模型 {model_name} 同樣失敗: {e_cpu}. 返回空嵌入字典。")
                return {}
        else: # 如果一開始就是 CPU 也失敗
            return {}

    # --- 創建前處理將欄位串接 ---
    # 使用欄位: title, abstract, category, subcategory
    # 加入 " [SEP] " 作為字串連接符
    news_embeddings_dict = {}
    news_df['combined_text'] = news_df['title'].astype(str) + " [SEP] " + \
                               news_df['abstract'].astype(str) + " [SEP] " + \
                               news_df['category'].astype(str) + " [SEP] " + \
                               news_df['subcategory'].astype(str)
    texts_to_encode = news_df['combined_text'].tolist()
    news_ids = news_df['news_id'].tolist()
    logger.info(f"共有 {len(texts_to_encode)} 篇新聞待編碼。")

    batch_size_embed = 256 # 可以根據VRAM調整 SentenceTransformer 的批次大小
    all_embeddings = []
    for i in tqdm(range(0, len(texts_to_encode), batch_size_embed), desc=f"生成新聞嵌入 ({model_name})"):
        batch_texts = texts_to_encode[i:i + batch_size_embed]
        try:
            embeddings = model.encode(batch_texts, convert_to_tensor=False, show_progress_bar=False) # 直接輸出 numpy array
            all_embeddings.append(embeddings) # embeddings 已经是 numpy array
        except torch.cuda.OutOfMemoryError:
            logger.error(f"CUDA out of memory during embedding generation with batch size {batch_size_embed}. Trying with smaller batch size or on CPU.")
            # 這裡可以選擇降級到CPU或進一步減小批次，或直接報錯退出
            # 為簡單起見，這裡報錯
            if DEVICE != torch.device('cpu'):
                logger.warning("嘗試在 CPU 上重新生成此批次...")
                try:
                    temp_model_cpu = SentenceTransformer(model_name, device='cpu')
                    embeddings = temp_model_cpu.encode(batch_texts, convert_to_tensor=False, show_progress_bar=False)
                    all_embeddings.append(embeddings)
                    del temp_model_cpu # 釋放記憶體
                    torch.cuda.empty_cache() # 清理CUDA快取
                except Exception as e_cpu_batch:
                    logger.error(f"在 CPU 上生成此批次嵌入失敗: {e_cpu_batch}")
                    return {} # 或者跳過此批次
            else:
                return {}

    if not all_embeddings:
        logger.error("沒有生成任何嵌入向量。請檢查新聞資料和模型。")
        return {}

    # 將 NumPy 陣列 垂直堆疊成一個單一的 NumPy 陣列
    all_embeddings_np = np.vstack(all_embeddings)
    # news_embeddings_dict 是一個字典，news_id作為KEY，將對應的將 embedding (VALUE) 存入字典中
    for news_id, embedding in zip(news_ids, all_embeddings_np):
        news_embeddings_dict[news_id] = embedding.astype(np.float32) # 確保是 float32

    logger.info(f"新聞嵌入生成完畢。共 {len(news_embeddings_dict)} 個嵌入向量。")
    if news_ids:
        logger.info(f"範例嵌入維度: {news_embeddings_dict[news_ids[0]].shape}")
        if news_embeddings_dict[news_ids[0]].shape[0] != NEWS_EMBEDDING_DIM:
            logger.error(f"生成的嵌入維度 ({news_embeddings_dict[news_ids[0]].shape[0]}) "
                         f"與模型預期維度 ({NEWS_EMBEDDING_DIM}) 不符。")
            return {}
    else:
        logger.warning("新聞 ID 列表為空，無法檢查嵌入維度。")


    logger.info(f"正在儲存新聞嵌入至快取檔案: {cache_path}")
    try:
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        with open(cache_path, 'wb') as f:
            pickle.dump(news_embeddings_dict, f)
        logger.info("新聞嵌入已成功儲存。")
    except Exception as e:
        logger.error(f"儲存新聞嵌入失敗: {e}")

    return news_embeddings_dict

# --- 3. 行為資料處理與特徵生成 ---
def get_user_history_embedding(clicked_news_history_str, news_embeddings_dict):
    if pd.isna(clicked_news_history_str) or clicked_news_history_str == '':
        return np.zeros(NEWS_EMBEDDING_DIM, dtype=np.float32)
    clicked_news_ids = clicked_news_history_str.split(' ')
    history_embeddings = []
    for news_id in clicked_news_ids:
        if news_id in news_embeddings_dict:
            history_embeddings.append(news_embeddings_dict[news_id])
    if not history_embeddings:
        return np.zeros(NEWS_EMBEDDING_DIM, dtype=np.float32)
    # 使用 float32 進行平均計算以節省記憶體並匹配模型期望的類型
    return np.mean(history_embeddings, axis=0).astype(np.float32)


def create_interaction_dataset(behaviors_df, news_embeddings_dict, is_train=True):
    logger.info(f"開始創建互動資料集 (is_train={is_train})...")
    processed_data = []

    for idx, row in tqdm(behaviors_df.iterrows(), total=len(behaviors_df), desc="處理行為資料"):
        clicked_news_history_str = row['clicked_news_history']
        impressions_str = row['impressions']

        user_history_embedding = get_user_history_embedding(clicked_news_history_str, news_embeddings_dict)

        if is_train:
            impression_news_with_labels = impressions_str.split(' ')
            for item in impression_news_with_labels:
                parts = item.split('-')
                if len(parts) == 2:
                    news_id, clicked_label_str = parts
                    try:
                        clicked_label = int(clicked_label_str)
                        if news_id in news_embeddings_dict:
                            candidate_news_embedding = news_embeddings_dict[news_id]
                            processed_data.append({
                                'user_history_embedding': user_history_embedding,
                                'candidate_news_embedding': candidate_news_embedding,
                                'label': clicked_label
                            })
                        # else: logger.debug(f"訓練資料中，新聞 ID {news_id} 無嵌入，跳過。") # 可能產生過多log
                    except ValueError:
                         logger.warning(f"訓練資料中發現無法轉換為int的label: {clicked_label_str} for item {item} in row index {idx}. 跳過此項。")
                # else: logger.warning(f"訓練資料中發現格式不符的 impression item: {item} ... 跳過。") # 可能產生過多log
        else: # is_test
            submission_id = row['id']
            impression_news_ids_raw = impressions_str.split(' ')

            impression_samples_for_this_user = []
            news_ids_in_impression_to_rank = [] # 用於Kaggle提交格式排序

            for news_id_in_impression in impression_news_ids_raw:
                clean_news_id = news_id_in_impression.split('-')[0] # 測試集通常不帶 "-label"
                if clean_news_id in news_embeddings_dict:
                    candidate_news_embedding = news_embeddings_dict[clean_news_id]
                else:
                    # logger.warning(f"測試資料中，新聞 ID {clean_news_id} (for ID {submission_id}) 未在嵌入字典中找到。使用零向量。")
                    candidate_news_embedding = np.zeros(NEWS_EMBEDDING_DIM, dtype=np.float32)

                impression_samples_for_this_user.append({
                    'user_history_embedding': user_history_embedding,
                    'candidate_news_embedding': candidate_news_embedding,
                })
                news_ids_in_impression_to_rank.append(clean_news_id) # 記錄原始新聞ID順序

            # 測試集不需要截斷或填充到15個，模型會預測所有曝光新聞，然後由提交邏輯處理
            processed_data.append({
                'id': submission_id,
                'samples': impression_samples_for_this_user,
            })


    logger.info(f"互動資料集創建完成。")
    if is_train:
        logger.info(f"訓練/驗證用樣本數: {len(processed_data)}")
        return pd.DataFrame(processed_data)
    else:
        logger.info(f"測試用提交組數: {len(processed_data)}")
        return processed_data

# --- 4. PyTorch Dataset 和 DataLoader ---
class NewsRecommendationDataset(Dataset):
    def __init__(self, data_list, is_test=False): # 新增 is_test 參數
        self.data_list = data_list
        self.is_test = is_test

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        item = self.data_list[idx]
        if self.is_test: # 測試集樣本結構不同
             return {
                'user_history_embedding': torch.tensor(item['user_history_embedding'], dtype=torch.float),
                'candidate_news_embedding': torch.tensor(item['candidate_news_embedding'], dtype=torch.float),
            }
        else: # 訓練/驗證集
            return {
                'user_history_embedding': torch.tensor(item['user_history_embedding'], dtype=torch.float),
                'candidate_news_embedding': torch.tensor(item['candidate_news_embedding'], dtype=torch.float),
                'label': torch.tensor(item['label'], dtype=torch.float)
            }

# --- 5. 模型建構 ---
class FFNRecommender(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim=1, dropout_rate=0.3):
        super(FFNRecommender, self).__init__()
        layers = []
        current_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(current_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            current_dim = h_dim
        layers.append(nn.Linear(current_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, user_history_emb, candidate_news_emb):
        # 確保輸入是2D: (batch_size, feature_dim)
        if user_history_emb.ndim == 1: user_history_emb = user_history_emb.unsqueeze(0)
        if candidate_news_emb.ndim == 1: candidate_news_emb = candidate_news_emb.unsqueeze(0)

        x = torch.cat((user_history_emb, candidate_news_emb), dim=1)
        return self.network(x)

# --- 6. 訓練與評估 ---
# 礬土中文註解：train_model 函數現在接受 model_save_path 作為參數，使其可以在 Optuna trial 中儲存特定 trial 的模型
def train_model(model, train_loader, val_loader, optimizer, criterion, epochs, device, model_save_path, patience, trial_num=None): # 礬土中文註解：新增 trial_num 參數用於日誌記錄
    logger.info(f"開始模型訓練... (Trial: {trial_num if trial_num is not None else 'N/A'})")
    best_val_auc = 0.0
    epochs_no_improve = 0
    history = {'train_loss': [], 'val_loss': [], 'val_auc': []}

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        # 使用了 disable=None 来允许TQDM的全局禁用/启用设置生效
        train_pbar_desc = f"Epoch {epoch+1}/{epochs} [訓練]"
        if trial_num is not None:
            train_pbar_desc = f"T{trial_num} " + train_pbar_desc
        train_pbar = tqdm(train_loader, desc=train_pbar_desc, leave=True, disable=False)
        for batch_idx, batch in enumerate(train_pbar):
            user_hist_emb = batch['user_history_embedding'].to(device)
            candidate_emb = batch['candidate_news_embedding'].to(device)
            labels = batch['label'].unsqueeze(1).to(device)

            optimizer.zero_grad()
            outputs = model(user_hist_emb, candidate_emb)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
            if batch_idx % 100 == 0: # 每100個batch更新一次進度條的loss顯示
                 train_pbar.set_postfix({'loss': loss.item()})
        avg_train_loss = total_train_loss / len(train_loader)
        history['train_loss'].append(avg_train_loss)

        model.eval()
        total_val_loss = 0
        all_labels = []
        all_predictions = []
        val_pbar_desc = f"Epoch {epoch+1}/{epochs} [驗證]"
        if trial_num is not None:
            val_pbar_desc = f"T{trial_num} " + val_pbar_desc
        val_pbar = tqdm(val_loader, desc=val_pbar_desc, leave=True, disable=False)
        with torch.no_grad():
            for batch_idx, batch in enumerate(val_pbar):
                user_hist_emb = batch['user_history_embedding'].to(device)
                candidate_emb = batch['candidate_news_embedding'].to(device)
                labels = batch['label'].unsqueeze(1).to(device)
                outputs = model(user_hist_emb, candidate_emb)
                loss = criterion(outputs, labels)
                total_val_loss += loss.item()
                predictions = torch.sigmoid(outputs)
                all_labels.extend(labels.cpu().numpy())
                all_predictions.extend(predictions.cpu().numpy())
                if batch_idx % 100 == 0:
                    val_pbar.set_postfix({'loss': loss.item()})

        avg_val_loss = total_val_loss / len(val_loader)
        # 確保 all_labels 和 all_predictions 不是空的
        if not all_labels or not all_predictions:
            logger.warning(f"Epoch {epoch+1}/{epochs} (Trial {trial_num}): 驗證集為空或未能生成預測。跳過AUC計算。")
            val_auc = 0.0 # 或者用之前的值
        else:
            try:
                val_auc = roc_auc_score(np.array(all_labels).flatten(), np.array(all_predictions).flatten())
            except ValueError as e:
                logger.error(f"計算AUC時出錯 (Trial {trial_num}): {e}. 可能標籤只有一個類別。")
                logger.info(f"Labels (Trial {trial_num}): {np.unique(np.array(all_labels).flatten(), return_counts=True)}")
                val_auc = 0.0 # 或者根據情況處理

        history['val_loss'].append(avg_val_loss)
        history['val_auc'].append(val_auc)
        logger.info(f"Epoch {epoch+1}/{epochs} (Trial {trial_num}): 訓練損失: {avg_train_loss:.4f}, 驗證損失: {avg_val_loss:.4f}, 驗證 AUC: {val_auc:.4f}")

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            epochs_no_improve = 0
            torch.save(model.state_dict(), model_save_path) # 礬土中文註解：儲存到指定的 model_save_path
            logger.info(f"發現新的最佳模型 (Trial {trial_num}), AUC: {best_val_auc:.4f}。模型已儲存至 {model_save_path}")
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            logger.info(f"連續 {patience} 個 epochs 驗證 AUC 未提升 (Trial {trial_num})，觸發早停。")
            break

    logger.info(f"訓練完成 (Trial {trial_num})。最佳驗證 AUC: {best_val_auc:.4f}")
    # 礬土中文註解：移除原來的最終模型訓練結果日誌，Optuna 將會有總結
    return history, best_val_auc


# 礬土中文註解：定義 Optuna 的 objective 函數
def objective(trial, train_s, val_s):
    logger.info(f"\n===== Optuna Trial {trial.number} 開始 =====")

    # 讓 Optuna 建議超參數
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [256, 512, 1024]) # 增加 2048 選項
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)

    # 建議隱藏層配置
    hidden_dims_options = {
        "config1": [512, 256],
        "config2": [512, 256, 128],
        "config3": [768, 384, 128],
        "config4": [1024, 512, 256],
        "config5": [256, 128]
    }
    selected_hidden_config_key = trial.suggest_categorical("hidden_dims_config", list(hidden_dims_options.keys()))
    hidden_dims = hidden_dims_options[selected_hidden_config_key]

    logger.info(f"Trial {trial.number} 超參數:")
    logger.info(f"  LEARNING_RATE: {lr}")
    logger.info(f"  BATCH_SIZE: {batch_size}")
    logger.info(f"  DROPOUT_RATE: {dropout_rate}")
    logger.info(f"  HIDDEN_DIMS: {hidden_dims}")

    # 為此 trial 創建 Dataset 和 DataLoader
    trial_train_dataset = NewsRecommendationDataset(train_s, is_test=False)
    trial_val_dataset = NewsRecommendationDataset(val_s, is_test=False)

    num_workers = 0 # 在 Windows 上 num_workers > 0 有時會有問題
    trial_train_loader = DataLoader(trial_train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True if DEVICE == torch.device('cuda') else False)
    trial_val_loader = DataLoader(trial_val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True if DEVICE == torch.device('cuda') else False)

    # 建立模型
    model = FFNRecommender(input_dim=MODEL_INPUT_DIM, hidden_dims=hidden_dims, dropout_rate=dropout_rate).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    # 設定此 trial 模型的儲存路徑
    trial_model_save_path = os.path.join(RESULT_DIR, f'model_trial_{trial.number}_auc.pth')

    # 訓練模型
    _, best_val_auc_trial = train_model(model, trial_train_loader, trial_val_loader, optimizer, criterion, EPOCHS_PER_TRIAL, DEVICE, trial_model_save_path, EARLY_STOPPING_PATIENCE, trial_num=trial.number)
    
    logger.info(f"===== Optuna Trial {trial.number} 結束, Best Val AUC: {best_val_auc_trial:.4f} =====")
    
    # Optuna 會嘗試最大化此返回值
    return best_val_auc_trial



#  behavior資料處理與特徵生成

In [ ]:
# --- 處理點擊新聞歷史字串 ---
# 將從 news_embeddings_dict 字典中 取出對應新聞編號的 embedding vector
def get_user_history_embedding(clicked_news_history_str, news_embeddings_dict):
    if pd.isna(clicked_news_history_str) or clicked_news_history_str == '':
        return np.zeros(NEWS_EMBEDDING_DIM, dtype=np.float32)
    # 用 ' ' 分割字串
    clicked_news_ids = clicked_news_history_str.split(' ')
    history_embeddings = []
    for news_id in clicked_news_ids:
        if news_id in news_embeddings_dict:
            history_embeddings.append(news_embeddings_dict[news_id])
    if not history_embeddings:
        return np.zeros(NEWS_EMBEDDING_DIM, dtype=np.float32)
    # 使用 float32 進行平均計算以節省記憶體並匹配模型期望的類型
    return np.mean(history_embeddings, axis=0).astype(np.float32)


# --- behavior資料處理與特徵生成 ---
def create_interaction_dataset(behaviors_df, news_embeddings_dict, is_train=True):
    logger.info(f"開始創建互動資料集 (is_train={is_train})...")
    processed_data = []

    for idx, row in tqdm(behaviors_df.iterrows(), total=len(behaviors_df), desc="處理行為資料"):
        clicked_news_history_str = row['clicked_news_history']
        impressions_str = row['impressions']

        user_history_embedding = get_user_history_embedding(clicked_news_history_str, news_embeddings_dict)

        if is_train:
            impression_news_with_labels = impressions_str.split(' ') # 處理 impressions
            for item in impression_news_with_labels:
                parts = item.split('-')  # N383574-0 -> ['N383574', '0']
                if len(parts) == 2:
                    news_id, clicked_label_str = parts
                    try:
                        clicked_label = int(clicked_label_str)
                        if news_id in news_embeddings_dict:
                            candidate_news_embedding = news_embeddings_dict[news_id] # 候選新聞嵌入向量
                            processed_data.append({
                                'user_history_embedding': user_history_embedding,     # 410559 N109405 N79284 N812877...的embedding vector 由 get_user_history_embedding()取得
                                'candidate_news_embedding': candidate_news_embedding, # N383574 的 embedding vector 由 news_embeddings_dict取得
                                'label': clicked_label                                # 0 (0或1)
                            })
                        # else: logger.debug(f"訓練資料中，新聞 ID {news_id} 無嵌入，跳過。") # 可能產生過多log
                    except ValueError:
                         logger.warning(f"訓練資料中發現無法轉換為int的label: {clicked_label_str} for item {item} in row index {idx}. 跳過此項。")
                # else: logger.warning(f"訓練資料中發現格式不符的 impression item: {item} ... 跳過。") # 可能產生過多log
        else: # is_test
            submission_id = row['id']
            impression_news_ids_raw = impressions_str.split(' ')

            impression_samples_for_this_user = []
            news_ids_in_impression_to_rank = [] # 用於Kaggle提交格式排序

            for news_id_in_impression in impression_news_ids_raw:
                clean_news_id = news_id_in_impression.split('-')[0] # 測試集通常不帶 "-label"
                if clean_news_id in news_embeddings_dict:
                    candidate_news_embedding = news_embeddings_dict[clean_news_id]
                else:
                    logger.warning(f"測試資料中，新聞 ID {clean_news_id} (for ID {submission_id}) 未在嵌入字典中找到。使用零向量。")
                    candidate_news_embedding = np.zeros(NEWS_EMBEDDING_DIM, dtype=np.float32)

                impression_samples_for_this_user.append({
                    'user_history_embedding': user_history_embedding,
                    'candidate_news_embedding': candidate_news_embedding,
                })
                news_ids_in_impression_to_rank.append(clean_news_id) # 記錄原始新聞ID順序

            # 測試集不需要截斷或填充到15個，每航都剛好是15個
            processed_data.append({
                'id': submission_id,
                'samples': impression_samples_for_this_user,
            })


    logger.info(f"互動資料集創建完成。")
    if is_train:
        logger.info(f"訓練/驗證用樣本數: {len(processed_data)}")
        return pd.DataFrame(processed_data)
    else:
        logger.info(f"測試用提交組數: {len(processed_data)}")
        return processed_data


# Dataset 和 DataLoader 

In [5]:

class NewsRecommendationDataset(Dataset):
    def __init__(self, data_list, is_test=False): # 新增 is_test 參數
        self.data_list = data_list
        self.is_test = is_test

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        item = self.data_list[idx]
        if self.is_test: # 測試集樣本結構不同
             return {
                'user_history_embedding': torch.tensor(item['user_history_embedding'], dtype=torch.float),
                'candidate_news_embedding': torch.tensor(item['candidate_news_embedding'], dtype=torch.float),
            }
        else: # 訓練/驗證集
            return {
                'user_history_embedding': torch.tensor(item['user_history_embedding'], dtype=torch.float),
                'candidate_news_embedding': torch.tensor(item['candidate_news_embedding'], dtype=torch.float),
                'label': torch.tensor(item['label'], dtype=torch.float)
            }


# 模型建構

In [6]:
class FFNRecommender(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim=1, dropout_rate=0.3):
        super(FFNRecommender, self).__init__()
        layers = []
        current_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(current_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            current_dim = h_dim
        layers.append(nn.Linear(current_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, user_history_emb, candidate_news_emb):
        # 確保輸入是2D: (batch_size, feature_dim)
        if user_history_emb.ndim == 1: user_history_emb = user_history_emb.unsqueeze(0)
        if candidate_news_emb.ndim == 1: candidate_news_emb = candidate_news_emb.unsqueeze(0)

        x = torch.cat((user_history_emb, candidate_news_emb), dim=1)
        return self.network(x)


# 訓練與評估

In [7]:
# --- 訓練與評估 ---
# train_model 函數現在接受 model_save_path 作為參數，使其可以在 Optuna trial 中儲存特定 trial 的模型
def train_model(model, train_loader, val_loader, optimizer, criterion, epochs, device, model_save_path, patience, trial_num=None): # trial_num 參數用於日誌記錄
    logger.info(f"開始模型訓練... (Trial: {trial_num if trial_num is not None else 'N/A'})")
    best_val_auc = 0.0
    epochs_no_improve = 0
    history = {'train_loss': [], 'val_loss': [], 'val_auc': []}

    
    for epoch in range(epochs):
        # --- 訓練  ---
        model.train()
        total_train_loss = 0
        # 使用了 disable=None 来允许TQDM的全局禁用/启用设置生效
        train_pbar_desc = f"Epoch {epoch+1}/{epochs} [訓練]"
        if trial_num is not None:
            train_pbar_desc = f"T{trial_num} " + train_pbar_desc
        train_pbar = tqdm(train_loader, desc=train_pbar_desc, leave=True, disable=False)
        for batch_idx, batch in enumerate(train_pbar):
            user_hist_emb = batch['user_history_embedding'].to(device)
            candidate_emb = batch['candidate_news_embedding'].to(device)
            labels = batch['label'].unsqueeze(1).to(device)

            optimizer.zero_grad()
            outputs = model(user_hist_emb, candidate_emb)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
            if batch_idx % 100 == 0: # 每100個batch更新一次進度條的loss顯示
                 train_pbar.set_postfix({'loss': loss.item()})
        avg_train_loss = total_train_loss / len(train_loader)
        history['train_loss'].append(avg_train_loss)

        # --- 驗證  ---
        model.eval()
        total_val_loss = 0
        all_labels = []
        all_predictions = []
        val_pbar_desc = f"Epoch {epoch+1}/{epochs} [驗證]"
        if trial_num is not None:
            val_pbar_desc = f"T{trial_num} " + val_pbar_desc
        val_pbar = tqdm(val_loader, desc=val_pbar_desc, leave=True, disable=False)
        with torch.no_grad():
            for batch_idx, batch in enumerate(val_pbar):
                user_hist_emb = batch['user_history_embedding'].to(device)
                candidate_emb = batch['candidate_news_embedding'].to(device)
                labels = batch['label'].unsqueeze(1).to(device)
                outputs = model(user_hist_emb, candidate_emb)
                loss = criterion(outputs, labels)
                total_val_loss += loss.item()
                predictions = torch.sigmoid(outputs)
                all_labels.extend(labels.cpu().numpy())
                all_predictions.extend(predictions.cpu().numpy())
                if batch_idx % 100 == 0:
                    val_pbar.set_postfix({'loss': loss.item()})

        avg_val_loss = total_val_loss / len(val_loader)
        # 確保 all_labels 和 all_predictions 不是空的
        if not all_labels or not all_predictions:
            logger.warning(f"Epoch {epoch+1}/{epochs} (Trial {trial_num}): 驗證集為空或未能生成預測。跳過AUC計算。")
            val_auc = 0.0 # 或者用之前的值
        else:
            try:
                val_auc = roc_auc_score(np.array(all_labels).flatten(), np.array(all_predictions).flatten())
            except ValueError as e:
                logger.error(f"計算AUC時出錯 (Trial {trial_num}): {e}. 可能標籤只有一個類別。")
                logger.info(f"Labels (Trial {trial_num}): {np.unique(np.array(all_labels).flatten(), return_counts=True)}")
                val_auc = 0.0 # 或者根據情況處理

        history['val_loss'].append(avg_val_loss)
        history['val_auc'].append(val_auc)
        logger.info(f"Epoch {epoch+1}/{epochs} (Trial {trial_num}): 訓練損失: {avg_train_loss:.4f}, 驗證損失: {avg_val_loss:.4f}, 驗證 AUC: {val_auc:.4f}")

        # 儲存最佳模型 以及 Trial 超參數
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            epochs_no_improve = 0
            torch.save(model.state_dict(), model_save_path) # 儲存到指定的 model_save_path
            logger.info(f"發現新的最佳模型 (Trial {trial_num}), AUC: {best_val_auc:.4f}。模型已儲存至 {model_save_path}")
        else:
            epochs_no_improve += 1

        # 早期停止: 連續n個EPOCHE的AUC沒有上升，就停止
        if epochs_no_improve >= patience:
            logger.info(f"連續 {patience} 個 epochs 驗證 AUC 未提升 (Trial {trial_num})，觸發早停。")
            break

    logger.info(f"訓練完成 (Trial {trial_num})。最佳驗證 AUC: {best_val_auc:.4f}")
    
    return history, best_val_auc



# 定義 Optuna 的 objective 函數

In [ ]:

# -- 定義 Optuna 的 objective 函數 --
def objective(trial, train_s, val_s):
    logger.info(f"\n===== Optuna Trial {trial.number} 開始 =====")

    # 讓 Optuna 建議超參數
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [256, 512, 1024]) # 可再增加 2048 選項
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)

    # 建議隱藏層配置
    hidden_dims_options = {
        "config1": [512, 256],
        "config2": [512, 256, 128],
        "config3": [768, 384, 128],
        "config4": [1024, 512, 256],
        "config5": [256, 128]
    }
 
    selected_hidden_config_key = trial.suggest_categorical("hidden_dims_config", list(hidden_dims_options.keys()))
    hidden_dims = hidden_dims_options[selected_hidden_config_key]

    logger.info(f"Trial {trial.number} 超參數:")
    logger.info(f"  LEARNING_RATE: {lr}")
    logger.info(f"  BATCH_SIZE: {batch_size}")
    logger.info(f"  DROPOUT_RATE: {dropout_rate}")
    logger.info(f"  HIDDEN_DIMS: {hidden_dims}")

    # 為此 trial 創建 Dataset 和 DataLoader
    trial_train_dataset = NewsRecommendationDataset(train_s, is_test=False)
    trial_val_dataset = NewsRecommendationDataset(val_s, is_test=False)

    num_workers = 0 # 在 Windows 上 num_workers > 0 有時會有問題
    trial_train_loader = DataLoader(trial_train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True if DEVICE == torch.device('cuda') else False)
    trial_val_loader = DataLoader(trial_val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True if DEVICE == torch.device('cuda') else False)

    # 建立模型
    model = FFNRecommender(input_dim=MODEL_INPUT_DIM, hidden_dims=hidden_dims, dropout_rate=dropout_rate).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    # 設定此 trial 模型的儲存路徑
    trial_model_save_path = os.path.join(RESULT_DIR, f'model_trial_{trial.number}_auc.pth')

    # 訓練模型
    _, best_val_auc_trial = train_model(model, trial_train_loader, trial_val_loader, optimizer, criterion, EPOCHS_PER_TRIAL, DEVICE, trial_model_save_path, EARLY_STOPPING_PATIENCE, trial_num=trial.number)
    
    logger.info(f"===== Optuna Trial {trial.number} 結束, Best Val AUC: {best_val_auc_trial:.4f} =====")
    
    # Optuna 會嘗試最大化此返回值
    return best_val_auc_trial



In [ ]:
# --- 7. 主執行流程 ---
if __name__ == '__main__':
    start_time = datetime.now()
    logger.info(f"腳本執行開始時間: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    logger.info(f"使用 PyTorch 裝置: {DEVICE}")
    logger.info(f"新聞嵌入模型: {NEWS_EMBEDDING_MODEL_NAME}, 維度: {NEWS_EMBEDDING_DIM}")

    logger.info("===== 開始新聞推薦預測任務 (Optuna 優化) =====")

    os.makedirs(TRAIN_DIR, exist_ok=True)
    os.makedirs(TEST_DIR, exist_ok=True)
    os.makedirs(RESULT_DIR, exist_ok=True)

    train_news_df = load_news_data(TRAIN_NEWS_PATH)
    test_news_df = load_news_data(TEST_NEWS_PATH)
    if train_news_df is None or test_news_df is None:
        logger.error("新聞資料載入失敗，終止執行。")
        sys.exit(1)

    all_news_df = pd.concat([train_news_df, test_news_df]).drop_duplicates(subset=['news_id']).reset_index(drop=True)
    logger.info(f"合併後的總新聞數 (去除重覆後): {len(all_news_df)}")

    # 明確傳遞模型名稱和快取路徑
    news_embeddings_dict = generate_news_embeddings_with_bge(
        all_news_df,
        model_name=NEWS_EMBEDDING_MODEL_NAME,
        cache_path=NEWS_EMBEDDINGS_CACHE_PATH
    )
    
    if news_embeddings_dict:
        sample_news_id = next(iter(news_embeddings_dict))
        logger.info(f"範例新聞嵌入維度: {news_embeddings_dict[sample_news_id].shape}")
        if news_embeddings_dict[sample_news_id].shape[0] != NEWS_EMBEDDING_DIM:
            logger.error(f"新聞嵌入維度 ({news_embeddings_dict[sample_news_id].shape[0]}) 與設定的 NEWS_EMBEDDING_DIM ({NEWS_EMBEDDING_DIM}) 不符。")
            sys.exit(1)
        else:
            logger.info(f"成功載入/生成 {len(news_embeddings_dict)} 個新聞嵌入。")
    else:
        logger.error("新聞嵌入字典為空，無法繼續。")
        sys.exit(1)

    logger.info("處理訓練用行為資料...")
    train_behaviors_df = load_behaviors_data(TRAIN_BEHAVIORS_PATH, is_train=True)
    if train_behaviors_df is None:
        logger.error("訓練行為資料載入失敗，終止執行。")
        sys.exit(1)

    # 未經過分割的完整資料集
    interaction_df_train_val = create_interaction_dataset(train_behaviors_df, news_embeddings_dict, is_train=True)
    if interaction_df_train_val.empty:
        logger.error("從訓練行為資料中未能創建有效的互動資料，終止執行。")
        sys.exit(1)
    logger.info(f"創建的訓練/驗證互動樣本數: {len(interaction_df_train_val)}")
    train_data_list_full = interaction_df_train_val.to_dict('records') #  Pandas DataFrame 轉換成一個字典列表
    del interaction_df_train_val # 釋放記憶體
    import gc
    gc.collect()


    if not train_data_list_full:
        logger.error("互動資料列表為空，無法進行訓練/驗證分割。")
        sys.exit(1)

    # --- train/val資料分割 ---
    try:
        # 將原始訓練資料分割為訓練集和驗證集，提供 Optuna objective 函數使用
        train_samples_list, val_samples_list = train_test_split(
            train_data_list_full,
            test_size=VALIDATION_SPLIT_RATIO,
            random_state=RANDOM_SEED,
            stratify=[d['label'] for d in train_data_list_full] # 參數用來確保訓練集和驗證集中 label 的分佈一致，防止某些類別在某一部分的資料集中過於稀缺或過於集中
        )
        logger.info("使用分層抽樣進行訓練/驗證集分割 (供 Optuna 使用)。")
    except ValueError:
        logger.warning("分層抽樣失敗 (可能因為某些類別樣本數過少)，改用普通隨機抽樣 (供 Optuna 使用)。")
        train_samples_list, val_samples_list = train_test_split(
            train_data_list_full,
            test_size=VALIDATION_SPLIT_RATIO,
            random_state=RANDOM_SEED  
        )
    del train_data_list_full
    gc.collect()

    logger.info(f"Optuna 用訓練樣本數: {len(train_samples_list)}")
    logger.info(f"Optuna 用驗證樣本數: {len(val_samples_list)}")

    if not train_samples_list or not val_samples_list:
        logger.error("Optuna 用的訓練集或驗證集為空，無法繼續。")
        sys.exit(1)

    # --- 訓練與驗證 ---
    # 執行 Optuna 超參數優化
    logger.info(f"===== 開始 Optuna 超參數優化 (Trials: {N_OPTUNA_TRIALS}, Epochs per trial: {EPOCHS_PER_TRIAL}) =====")
    study = optuna.create_study(direction='maximize')
    # 使用 lambda 將額外參數 (train_s, val_s) 傳遞給 objective 函數
    study.optimize(lambda trial: objective(trial, train_samples_list, val_samples_list), n_trials=N_OPTUNA_TRIALS)

    logger.info("===== Optuna 超參數優化結束 =====")
    logger.info(f"最佳 Trial 編號: {study.best_trial.number}")
    logger.info(f"最佳 Validation AUC: {study.best_value:.4f}")
    logger.info("最佳超參數:")
    for key, value in study.best_params.items():
        logger.info(f"  {key}: {value}")

    # 將 Optuna 找到的最佳模型的權重複製到最終的模型儲存路徑
    best_trial_model_path = os.path.join(RESULT_DIR, f'model_trial_{study.best_trial.number}_auc.pth')
    logger.info(f"將最佳 trial ({study.best_trial.number}) 的模型從 {best_trial_model_path} 複製到 {MODEL_SAVE_PATH}")
    try:
        shutil.copyfile(best_trial_model_path, MODEL_SAVE_PATH)
        logger.info(f"最佳模型已成功複製到 {MODEL_SAVE_PATH}")
    except Exception as e:
        logger.error(f"複製最佳模型失敗: {e}")
        logger.info("將嘗試直接使用最佳 trial 的模型權重路徑進行後續預測。")
        MODEL_SAVE_PATH = best_trial_model_path # 如果複製失敗，直接使用 trial 的路徑


    # 清理 Optuna trial 期間產生的資料和模型，只保留最佳的
    # (或者可以選擇保留所有 trial 的模型，這裡選擇清理)
    for i in range(N_OPTUNA_TRIALS):
        trial_model_file = os.path.join(RESULT_DIR, f'model_trial_{i}_auc.pth')
        if i != study.best_trial.number and os.path.exists(trial_model_file):
            try:
                os.remove(trial_model_file)
            except Exception as e:
                logger.warning(f"無法刪除 trial 模型檔案 {trial_model_file}: {e}")
    
    # 獲取最佳參數以供記錄或最終模型（如果需要重新訓練）
    best_hyperparameters = study.best_params
    final_learning_rate = best_hyperparameters['lr']
    final_batch_size = best_hyperparameters['batch_size']
    final_dropout_rate = best_hyperparameters['dropout_rate']
    # 再次從 options 中獲取 hidden_dims 列表
    hidden_dims_options_for_final = {
        "config1": [512, 256],
        "config2": [512, 256, 128],
        "config3": [768, 384, 128],
        "config4": [1024, 512, 256],
        "config5": [256, 128]
    }
    final_hidden_dims = hidden_dims_options_for_final[best_hyperparameters['hidden_dims_config']]


    # 使用 Optuna 找到的最佳參數配置模型，然後載入最佳權重
    model = FFNRecommender(
        input_dim=MODEL_INPUT_DIM,
        hidden_dims=final_hidden_dims, # 使用 Optuna 找到的最佳 hidden_dims
        dropout_rate=final_dropout_rate # 使用 Optuna 找到的最佳 dropout_rate
    ).to(DEVICE)

    logger.info("模型架構 (使用 Optuna 最佳參數):")
    logger.info(model)
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(f"模型可訓練參數總數: {total_params:,}")

    # 清理訓練資料相關記憶體
    del train_samples_list, val_samples_list
    gc.collect()
    if DEVICE == torch.device('cuda'):
        torch.cuda.empty_cache()


    logger.info("處理測試用行為資料並進行預測...")
    test_behaviors_df = load_behaviors_data(TEST_BEHAVIORS_PATH, is_train=False)
    if test_behaviors_df is None:
        logger.error("測試行為資料載入失敗，終止執行。")
        sys.exit(1)

    test_interaction_groups = create_interaction_dataset(test_behaviors_df, news_embeddings_dict, is_train=False)
    if not test_interaction_groups:
        logger.error("從測試行為資料中未能創建有效的互動資料，終止執行。")
        sys.exit(1)

    logger.info(f"載入 Optuna 找到的最佳模型: {MODEL_SAVE_PATH}")
    try:
        model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE))
    except FileNotFoundError:
        logger.error(f"模型檔案 {MODEL_SAVE_PATH} 未找到。請確保 Optuna 流程正確儲存了模型。")
        sys.exit(1)
    except Exception as e:
        logger.error(f"載入模型時發生錯誤: {e}")
        sys.exit(1)

    model.eval()

    # --- test預測 ---
    submission_results =[]
    test_pbar = tqdm(test_interaction_groups, desc="Kaggle 提交檔案預測", leave=True, disable=False)
    for group in test_pbar:
        impression_id = group['id']
        samples_for_impression = group['samples'] # 這是一個 list of dicts

        if not samples_for_impression:
            logger.error(f"Impression ID {impression_id} 沒有可預測的樣本。")
            sys.exit(1)
            
        user_hist_embs_list = [s['user_history_embedding'] for s in samples_for_impression]
        candidate_embs_list = [s['candidate_news_embedding'] for s in samples_for_impression]

        user_hist_embs_tensor = torch.tensor(np.array(user_hist_embs_list, dtype=np.float32)).to(DEVICE)
        candidate_embs_tensor = torch.tensor(np.array(candidate_embs_list, dtype=np.float32)).to(DEVICE)

        with torch.no_grad():
            outputs = model(user_hist_embs_tensor, candidate_embs_tensor)
            probabilities = torch.sigmoid(outputs).cpu().numpy().flatten()

        row_dict = {'id': impression_id}
        for i in range(15):
            row_dict[f'p{i+1}'] = probabilities[i]
        submission_results.append(row_dict)


    submission_df = pd.DataFrame(submission_results)
    cols = ['id'] + [f'p{i+1}' for i in range(15)]
    submission_df = submission_df[cols]
    
    
    # 生成csv提交文件
    try:
        submission_df.to_csv(SUBMISSION_PATH, index=False)
        logger.info(f"提交檔案已儲存至: {SUBMISSION_PATH}")
        logger.info("提交檔案預覽 (前5筆):")
        logger.info(submission_df.head().to_string(index=False))
    except Exception as e:
        logger.error(f"儲存提交檔案失敗: {e}")

    logger.info(f"--- Optuna 找到的最佳模型最終結果 ---")
    logger.info(f"最佳驗證 AUC (來自 Optuna): {study.best_value:.4f}")
    logger.info(f"使用的最佳超參數:")
    logger.info(f"  EMBEDDING_MODEL: {NEWS_EMBEDDING_MODEL_NAME}")
    logger.info(f"  NEWS_EMBEDDING_DIM: {NEWS_EMBEDDING_DIM}")
    logger.info(f"  LEARNING_RATE: {final_learning_rate}")
    logger.info(f"  BATCH_SIZE: {final_batch_size}")
    logger.info(f"  HIDDEN_DIMS: {final_hidden_dims}")
    logger.info(f"  DROPOUT_RATE: {final_dropout_rate}")

    logger.info("===== 新聞推薦預測任務結束 (Optuna 優化完成) =====")
    end_time = datetime.now()
    logger.info(f"腳本執行結束時間: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
    logger.info(f"總執行時間: {end_time - start_time}")


2025-06-06 18:25:02,819 - INFO - 腳本執行開始時間: 2025-06-06 18:25:02
2025-06-06 18:25:02,819 - INFO - 使用 PyTorch 裝置: cuda
2025-06-06 18:25:02,820 - INFO - 新聞嵌入模型: BAAI/bge-base-en-v1.5, 維度: 768
2025-06-06 18:25:02,820 - INFO - ===== 開始新聞推薦預測任務 (Optuna 優化) =====
2025-06-06 18:25:02,822 - INFO - 開始載入新聞資料: ../data/train\train_news.tsv
2025-06-06 18:25:03,555 - INFO - 新聞資料 train_news.tsv 載入完成，共 101527 筆記錄。
2025-06-06 18:25:03,555 - INFO - 開始載入新聞資料: ../data/test\test_news.tsv
2025-06-06 18:25:04,086 - INFO - 新聞資料 test_news.tsv 載入完成，共 72023 筆記錄。
2025-06-06 18:25:04,146 - INFO - 合併後的總新聞數 (去除重覆後): 104151
2025-06-06 18:25:04,148 - INFO - 開始使用 BAAI/bge-base-en-v1.5 生成新聞嵌入。這可能需要一些時間...
2025-06-06 18:25:04,149 - INFO - Load pretrained SentenceTransformer: BAAI/bge-base-en-v1.5
2025-06-06 18:25:08,047 - INFO - SentenceTransformer 模型 BAAI/bge-base-en-v1.5 已成功載入到 cuda。
2025-06-06 18:25:08,148 - INFO - 共有 104151 篇新聞待編碼。


生成新聞嵌入 (BAAI/bge-base-en-v1.5):   0%|          | 0/407 [00:00<?, ?it/s]

2025-06-06 18:28:24,112 - INFO - 新聞嵌入生成完畢。共 104151 個嵌入向量。
2025-06-06 18:28:24,112 - INFO - 範例嵌入維度: (768,)
2025-06-06 18:28:24,113 - INFO - 正在儲存新聞嵌入至快取檔案: ../result\news_embeddings_BAAI_bge-base-en-v1.5.pkl
2025-06-06 18:28:24,615 - INFO - 新聞嵌入已成功儲存。
2025-06-06 18:28:24,645 - INFO - 範例新聞嵌入維度: (768,)
2025-06-06 18:28:24,646 - INFO - 成功載入/生成 104151 個新聞嵌入。
2025-06-06 18:28:24,646 - INFO - 處理訓練用行為資料...
2025-06-06 18:28:24,647 - INFO - 開始載入行為資料: ../data/train\train_behaviors.tsv
2025-06-06 18:28:25,683 - INFO - 行為資料 train_behaviors.tsv 載入完成，共 285297 筆記錄。
2025-06-06 18:28:25,749 - INFO - 開始創建互動資料集 (is_train=True)...


處理行為資料:   0%|          | 0/285297 [00:00<?, ?it/s]

2025-06-06 18:28:50,453 - INFO - 互動資料集創建完成。
2025-06-06 18:28:50,454 - INFO - 訓練/驗證用樣本數: 4279455
2025-06-06 18:28:51,793 - INFO - 創建的訓練/驗證互動樣本數: 4279455
2025-06-06 18:28:56,956 - INFO - 使用分層抽樣進行訓練/驗證集分割 (供 Optuna 使用)。
2025-06-06 18:28:57,168 - INFO - Optuna 用訓練樣本數: 3423564
2025-06-06 18:28:57,168 - INFO - Optuna 用驗證樣本數: 855891
2025-06-06 18:28:57,169 - INFO - ===== 開始 Optuna 超參數優化 (Trials: 1, Epochs per trial: 1) =====
[I 2025-06-06 18:28:57,170] A new study created in memory with name: no-name-01036e22-6e8d-4604-9940-2395256cb161
2025-06-06 18:28:57,170 - INFO - 
===== Optuna Trial 0 開始 =====
2025-06-06 18:28:57,171 - INFO - Trial 0 超參數:
2025-06-06 18:28:57,171 - INFO -   LEARNING_RATE: 4.1208138932763944e-05
2025-06-06 18:28:57,172 - INFO -   BATCH_SIZE: 256
2025-06-06 18:28:57,173 - INFO -   DROPOUT_RATE: 0.42706370758034395
2025-06-06 18:28:57,173 - INFO -   HIDDEN_DIMS: [768, 384, 128]
2025-06-06 18:28:57,186 - INFO - 開始模型訓練... (Trial: 0)


...

2025-06-06 18:31:21,285 - INFO - Epoch 1/1 (Trial 0): 訓練損失: 0.3410, 驗證損失: 0.3236, 驗證 AUC: 0.7302
2025-06-06 18:31:21,297 - INFO - 發現新的最佳模型 (Trial 0), AUC: 0.7302。模型已儲存至 ../result\model_trial_0_auc.pth
2025-06-06 18:31:21,298 - INFO - 訓練完成 (Trial 0)。最佳驗證 AUC: 0.7302
2025-06-06 18:31:21,344 - INFO - ===== Optuna Trial 0 結束, Best Val AUC: 0.7302 =====
[I 2025-06-06 18:31:21,345] Trial 0 finished with value: 0.7302260154105821 and parameters: {'lr': 4.1208138932763944e-05, 'batch_size': 256, 'dropout_rate': 0.42706370758034395, 'hidden_dims_config': 'config1'}. Best is trial 0 with value: 0.7302260154105821.
2025-06-06 18:31:21,346 - INFO - ===== Optuna 超參數優化結束 =====
2025-06-06 18:31:21,347 - INFO - 最佳 Trial 編號: 0
2025-06-06 18:31:21,348 - INFO - 最佳 Validation AUC: 0.7302
2025-06-06 18:31:21,348 - INFO - 最佳超參數:
2025-06-06 18:31:21,348 - INFO -   lr: 4.1208138932763944e-05
2025-06-06 18:31:21,349 - INFO -   batch_size: 256
2025-06-06 18:31:21,350 - INFO -   dropout_rate: 0.42706370758034395

處理行為資料:   0%|          | 0/46332 [00:00<?, ?it/s]

2025-06-06 18:31:27,032 - INFO - 互動資料集創建完成。
2025-06-06 18:31:27,032 - INFO - 測試用提交組數: 46332
2025-06-06 18:31:27,033 - INFO - 載入 Optuna 找到的最佳模型: ../result\best_model_auc_deep_learning.pth


Kaggle 提交檔案預測:   0%|          | 0/46332 [00:00<?, ?it/s]

2025-06-06 18:31:57,742 - INFO - 提交檔案已儲存至: ../result\submission_deep_learning_optuna.csv
2025-06-06 18:31:57,743 - INFO - 提交檔案預覽 (前5筆):
2025-06-06 18:31:57,746 - INFO -  id       p1       p2       p3       p4       p5       p6       p7       p8       p9      p10      p11      p12      p13      p14      p15
  0 0.260294 0.073103 0.186649 0.326422 0.145776 0.336128 0.178001 0.110423 0.118146 0.080872 0.133643 0.238733 0.130766 0.108635 0.241649
  1 0.166369 0.112131 0.065277 0.150887 0.207354 0.126234 0.101619 0.114362 0.174965 0.269970 0.137129 0.151367 0.103892 0.145774 0.102767
  2 0.129345 0.382310 0.166545 0.045136 0.311192 0.100600 0.118459 0.062917 0.152969 0.301488 0.118858 0.198889 0.055398 0.063156 0.080814
  3 0.207369 0.089422 0.165093 0.152771 0.216001 0.267772 0.132473 0.058123 0.084116 0.372205 0.125974 0.170545 0.248964 0.377735 0.466265
  4 0.065405 0.254303 0.058720 0.233557 0.172421 0.183851 0.263894 0.184980 0.254803 0.100186 0.102734 0.209744 0.414761 0.269939 0.2155